# /news 端点连通性测试 (Client REST API)

走本地 RIT Client，不是 DMA 服务器。**跑之前 RIT Client 必须已启动并登录。**

跟 DMA 的区别只有两点：认证用 `X-API-Key` 而不是 Basic auth；限流由 Client 缓冲，基本不会吃 429。端点和返回格式完全一样。

In [ ]:
import requests

API_ENDPOINT = "http://localhost:9999/v1"
AUTHORIZATION = {"X-API-Key": "Rotman"}

session = requests.Session()
session.headers.update(AUTHORIZATION)

In [ ]:
# 1. 连通性 + 认证。报 ConnectionError = Client 没开；401 = API key 不对。
resp = session.get(f"{API_ENDPOINT}/case")
print(resp.status_code)
print(resp.json())

In [ ]:
# 2. 第一次拉新闻，不带游标
resp = session.get(f"{API_ENDPOINT}/news", params={"limit": 20})
print(resp.status_code)
news = resp.json()
news

In [ ]:
# 3. 核对字段名是否为 news_id / period / tick / ticker / headline / body
if news:
    print(sorted(news[0].keys()))
    print(news[0])

In [ ]:
# 4. 增量拉取：只要 news_id 比游标大的
last_news_id = max(n["news_id"] for n in news) if news else 0
print("last_news_id =", last_news_id)

resp2 = session.get(f"{API_ENDPOINT}/news", params={"after": last_news_id, "limit": 20})
print(resp2.status_code)
resp2.json()

In [ ]:
# 5. 用真实公告文本验证波动率解析
from vol_strategy import parse_vol_from_news

for n in news:
    print(n.get("tick"), "|", n.get("headline"), "->", parse_vol_from_news(n))

In [ ]:
# 6. 核对 /securities 的字段名（build_signal_table 依赖 ticker/last/bid/ask/position）
sec = session.get(f"{API_ENDPOINT}/securities").json()
print(sorted(sec[0].keys()))
sec[:3]